In [6]:
import pandas as pd
import numpy as np
import warnings
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, matthews_corrcoef,
                             log_loss, cohen_kappa_score, brier_score_loss)

# These must be installed in your environment:
# !pip install imbalanced-learn xgboost lightgbm
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, GRU, Bidirectional, Conv1D, GlobalAveragePooling1D, Reshape, Input, Dropout

warnings.filterwarnings('ignore')

# ==========================================
# 1. DATA PREPARATION (DATASET 2)
# ==========================================
df = pd.read_csv('file2.csv')

# Drop Target-related leakages and IDs
drop_cols = ['UDI', 'Product ID', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF', 'Machine failure']
X = df.drop(columns=[c for c in drop_cols if c in df.columns])
y = df['Machine failure'].astype(int)

# Preprocessing Pipeline
numeric_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('scal', StandardScaler())]), numeric_features),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore'))]), categorical_features)
])

# ==========================================
# 2. DEEP LEARNING MODEL BUILDER
# ==========================================
def build_dl_model(name, input_dim):
    model = Sequential([Input(shape=(input_dim,))])
    if name == 'MLP':
        model.add(Dense(64, activation='relu'))
        model.add(Dropout(0.2))
        model.add(Dense(32, activation='relu'))
    elif name == '1D-CNN':
        model.add(Reshape((input_dim, 1)))
        model.add(Conv1D(64, kernel_size=3, activation='relu'))
        model.add(GlobalAveragePooling1D())
    elif name in ['LSTM', 'GRU', 'BiLSTM']:
        model.add(Reshape((input_dim, 1)))
        if name == 'LSTM': model.add(LSTM(64))
        elif name == 'GRU': model.add(GRU(64))
        elif name == 'BiLSTM': model.add(Bidirectional(LSTM(64)))

    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy')
    return model

# ==========================================
# 3. EVALUATION LOOP (12 MODELS x 10 METRICS)
# ==========================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model_names = ['LR', 'NB', 'SVM', 'DT', 'RF', 'XGB', 'LGBM', 'MLP', '1D-CNN', 'LSTM', 'GRU', 'BiLSTM']
final_results = []

for name in model_names:
    print(f"Processing Model: {name}...")
    # Initialize 10 metrics
    m_store = {k: [] for k in ['acc', 'f1', 'prec', 'rec', 'spec', 'auc', 'mcc', 'kappa', 'brier', 'logloss']}

    for train_idx, val_idx in skf.split(X, y):
        Xt_raw, Xv_raw = X.iloc[train_idx], X.iloc[val_idx]
        yt, yv = y.iloc[train_idx], y.iloc[val_idx]

        # Preprocess and Oversample
        Xt_p = preprocessor.fit_transform(Xt_raw)
        Xv_p = preprocessor.transform(Xv_raw)
        X_res, y_res = SMOTE(random_state=42).fit_resample(Xt_p, yt)

        # Instantiate and Train
        if name == 'LR': clf = LogisticRegression(max_iter=1000).fit(X_res, y_res)
        elif name == 'NB': clf = GaussianNB().fit(X_res, y_res)
        elif name == 'SVM': clf = SVC(probability=True).fit(X_res, y_res)
        elif name == 'DT': clf = DecisionTreeClassifier().fit(X_res, y_res)
        elif name == 'RF': clf = RandomForestClassifier().fit(X_res, y_res)
        elif name == 'XGB': clf = XGBClassifier(eval_metric='logloss').fit(X_res, y_res)
        elif name == 'LGBM': clf = LGBMClassifier(verbose=-1).fit(X_res, y_res)
        else:
            clf = build_dl_model(name, X_res.shape[1])
            clf.fit(X_res, y_res, epochs=10, batch_size=32, verbose=0)

        # Predictions
        if name in ['MLP', '1D-CNN', 'LSTM', 'GRU', 'BiLSTM']:
            y_prob = clf.predict(Xv_p).flatten()
        else:
            y_prob = clf.predict_proba(Xv_p)[:, 1]

        y_pred = (y_prob > 0.5).astype(int)
        tn, fp, fn, tp = confusion_matrix(yv, y_pred).ravel()

        # Log all 10 Metrics
        m_store['acc'].append(accuracy_score(yv, y_pred))
        m_store['f1'].append(f1_score(yv, y_pred))
        m_store['prec'].append(precision_score(yv, y_pred))
        m_store['rec'].append(recall_score(yv, y_pred))
        m_store['spec'].append(tn / (tn + fp))
        m_store['auc'].append(roc_auc_score(yv, y_prob))
        m_store['mcc'].append(matthews_corrcoef(yv, y_pred))
        m_store['kappa'].append(cohen_kappa_score(yv, y_pred))
        m_store['brier'].append(brier_score_loss(yv, y_prob))
        m_store['logloss'].append(log_loss(yv, y_prob))

    # Compile Row (Mean ± Std)
    row = {'Model': name}
    for k in m_store:
        row[k] = f"{np.mean(m_store[k]):.4f} ± {np.std(m_store[k]):.4f}"
    final_results.append(row)

# ==========================================
# 4. FINAL OUTPUT
# ==========================================
summary_df = pd.DataFrame(final_results)
# Rename columns for clarity in paper
summary_df.columns = ['Model', 'Accuracy', 'F1', 'Precision', 'Recall', 'Specificity', 'AUC', 'MCC', 'Kappa', 'Brier', 'LogLoss']

print("\n" + "="*100)
print("DATASET 2: FULL BASELINE RESULTS (ALL MODELS & ALL METRICS)")
print("="*100)
print(summary_df.to_string(index=False))

# Export to CSV for your results section
summary_df.to_csv('baseline_complete_dataset2.csv', index=False)

Processing Model: LR...
Processing Model: NB...
Processing Model: SVM...
Processing Model: DT...
Processing Model: RF...
Processing Model: XGB...
Processing Model: LGBM...
Processing Model: MLP...
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Processing Model: 1D-CNN...
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Processing Model: LSTM...
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
Processing Model: GRU...
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step
63/63 ━━━━━━━━━━━━━━━━